# Assigment 3

In [50]:
import igl
import numpy as np
import pyvista as pv
pv.set_jupyter_backend('trame')

In [51]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

def plot_mesh_with_normals(V, F, N, factor=0.1):
    arrows = pv.vector_poly_data(V, N)
    arrows = arrows.glyph(orient='vectors',scale='mag',factor=factor)

    p = pv.Plotter()
    p.add_mesh(to_pyvista_mesh(V, F), show_edges=True)
    p.add_mesh(arrows)
    p.show()

def plot_lines(p, v0, v1, color):
    tmp = np.zeros((v0.shape[0] * 2, 3))
    tmp[0::2] = v0
    tmp[1::2] = v1
    p.add_lines(tmp, color=color)

In [52]:
v, f = igl.read_triangle_mesh("data/bunny.off")
to_pyvista_mesh(v, f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:58646/index.html?ui=P_0x11cc6cfef80_7&reconnect=auto" class="pyvis…

# Vertex normal

In [53]:
#Standard face normal --> I assume this is the standard vertex normals as described in README
vertexNormal = igl.per_vertex_normals(v, f, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_UNIFORM)
plot_mesh_with_normals(v, f, vertexNormal, factor=0.01)


Widget(value='<iframe src="http://localhost:58646/index.html?ui=P_0x11cafc5b130_8&reconnect=auto" class="pyvis…

In [54]:
#Area-weighted face normal
faceareaVertexNormal = igl.per_vertex_normals(v, f, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA)
plot_mesh_with_normals(v, f, faceareaVertexNormal, factor=0.01)


Widget(value='<iframe src="http://localhost:58646/index.html?ui=P_0x11cbc30ea70_9&reconnect=auto" class="pyvis…

In [55]:
#Mean-curvature normal
#note to self: reminder to use the cotangent-weighted Laplacian as described in readme
eps = 1e-8                                  #not told what to use, so pick a threshold value!

cotanLaplace = igl.cotmatrix(v,f)           #the cotangent Laplacian

verticesWithCotanLaplace = cotanLaplace @ v #apply to verteces 

meanCurveMag = np.linalg.norm(verticesWithCotanLaplace, axis=1)#magnitude of mean-curvature normal vectors

meanNormals = np.copy(faceareaVertexNormal)       #for sign comparison

reliableNormalsOnly = meanCurveMag > eps
meanNormals[reliableNormalsOnly] = verticesWithCotanLaplace[reliableNormalsOnly] / meanCurveMag[reliableNormalsOnly][:, None]

#we can get negative values, so we compare to standard normals to get correct direction
signs = np.sign(np.sum(meanNormals * vertexNormal, axis=1))
signs[signs == 0] = 1
meanNormals = meanNormals * signs[:, None]
plot_mesh_with_normals(v, f, meanNormals, factor=0.01)


Widget(value='<iframe src="http://localhost:58646/index.html?ui=P_0x11cbc99f370_10&reconnect=auto" class="pyvi…

In [56]:
#PCA normal


In [57]:
#Quadratic fitting normal


# Curvature

In [58]:
#gaussian curvature

In [59]:
# principal curvature

# Smoothing with the Laplacian

In [60]:
from scipy.sparse.linalg import spsolve
import scipy.sparse as sp

In [61]:
# Explicit laplacian

In [62]:
# Implicit laplacian
